# NB_lab_test_trends

Builds `adb_healthy.gold.lab_test_trends` (one row per test type per calendar month) from `silver.lab_results`, with optional calendar enrichment and a LAG()-based month-over-month abnormal-rate trend. Full rebuild on every run.

Notes:
* `silver.lab_results` is deduplicated on `lab_result_id` (latest `update_timestamp` kept) before aggregation.
* The quarter label is derived from `silver.calendar.quarter` as `'Q' || calendar.quarter` (e.g. "Q1").

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_healthy.gold;

In [ ]:
%sql
CREATE OR REPLACE TABLE adb_healthy.gold.lab_test_trends AS
WITH lab_results_dedup AS (
  SELECT *
  FROM adb_healthy.silver.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC) = 1
),
monthly AS (
  SELECT
    test_name,
    year(test_date)                          AS year,
    month(test_date)                         AS month,
    COUNT(*)                                 AS total_tests,
    COUNT_IF(result_status = 'Abnormal')     AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')     AS critical_test_count
  FROM lab_results_dedup
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(ROUND(abnormal_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name                                                     AS test_name,
  CAST(w.year AS INT)                                             AS year,
  CAST(w.month AS INT)                                            AS month,
  c.month_name                                                    AS month_name,
  concat('Q', c.quarter)                                          AS quarter_name,
  CAST(w.total_tests AS INT)                                      AS total_tests,
  CAST(w.abnormal_test_count AS INT)                              AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                              AS critical_test_count,
  w.abnormal_rate_pct                                             AS abnormal_rate_pct,
  w.critical_rate_pct                                             AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))            AS prev_month_abnormal_rate_pct,
  CAST(
    CASE WHEN w.prev_month_abnormal_rate_pct IS NULL THEN NULL
         ELSE w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct
    END AS DECIMAL(6,2))                                          AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN adb_healthy.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT);

In [ ]:
updated_df = spark.table("adb_healthy.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))